# Single-Bus Substation

The single-bus configuration is the simplest possible design, with the lowest cost and also the lowest reliability. All equipment and switching devices are connected to a single main bus, which is always energized, as shown below. This configuration is commonly used in small electric distribution substations and wind farm collectors due to its simplicity and lower cost. Bus faults and breaker failures will result in an outage or significant voltage deviation to all branches in the entire substation as all branches share a common bus that will propagate the problem to said branches

## CIM Representation

In CIM, all buses and junctions are represented by the ConnectivityNode class. If a node corresponds to a bus bar, then a BusBarSection object is appended to the ConnectivityNode, as shown below in Figure 2. The single-bus configuration only includes a single main bus, with all distribution feeders connected to said bus. Full node-breaker switch representation adds a set of one Breaker and two Disconnector objects for each feeder added to the substation.

![single-bus](../images/single_bus.png)

In [1]:
# Import transmission and distribution modeling classes
from cimgraph.models import FeederModel, NodeBreakerModel
from cimgraph.databases import ConnectionParameters, RDFlibConnection, BlazegraphConnection
import cimgraph.utils as utils

import importlib

In [2]:
cim_profile = 'cimhub_2023'
cim = importlib.import_module('cimgraph.data_profile.' + cim_profile)

In [3]:
params = ConnectionParameters(filename=None, cim_profile=cim_profile, iec61970_301=8)
connection = RDFlibConnection(params)

In [4]:
from cimbuilder.substation_builder import SingleBusSubstation

ModuleNotFoundError: No module named 'cimbuilder'

In [ ]:
SubBuilder = SingleBusSubstation(connection=connection, name="single_sub", base_voltage=115000)
substation = SubBuilder.substation


In [ ]:
# Import 13 bus model from XML file
ieee13_feeder = cim.Feeder(mRID = '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
params = ConnectionParameters(filename='../test_models/IEEE13.xml', cim_profile=cim_profile, iec61970_301=8)
connection = RDFlibConnection(params)
ieee13_network = FeederModel(connection=connection, container=ieee13_feeder, distributed=False)

In [ ]:

assets13_feeder = cim.Feeder(mRID = '5B816B93-7A5F-B64C-8460-47C17D6E4B0F')
params = ConnectionParameters(filename='../test_models/IEEE13_Assets.xml', cim_profile=cim_profile, iec61970_301=8)
connection = RDFlibConnection(params)
assets13_network = FeederModel(connection=connection, container=assets13_feeder, distributed=False)

In [ ]:
SubBuilder.new_feeder(series_number= 10, feeder=ieee13_feeder, feeder_network=ieee13_network)
SubBuilder.new_feeder(series_number = 20, feeder=assets13_feeder, feeder_network=assets13_network)

In [ ]:
SubBuilder.network.pprint(cim.Substation)

In [ ]:
SubBuilder.network.pprint(cim.Feeder)

In [ ]:
utils.write_xml(SubBuilder.network, '../test_output/single_bus.xml')

In [ ]:
# Connect to Blazegraph Database
from cimgraph.databases import BlazegraphConnection
params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql", cim_profile=cim_profile, iec61970_301=8)
blazegraph = BlazegraphConnection(params)
blazegraph.execute('drop all')

In [ ]:
from cimloader.databases.blazegraph import BlazegraphConnection as BlazegraphLoader
params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql", cim_profile=cim_profile, iec61970_301=8)
loader = BlazegraphLoader(params)

In [ ]:
loader.upload_from_file(filename='../test_models/IEEE13.xml')
loader.upload_from_file(filename='../test_models/IEEE13_Assets.xml')
loader.upload_from_file(filename='../test_output/single_bus.xml')

In [ ]:
network = NodeBreakerModel(container=substation, connection=blazegraph, distributed = False)

In [ ]:
# Print substation info
network.get_all_edges(cim.Substation)
network.pprint(cim.Substation)

In [ ]:
# Print feeder info
network.get_all_edges(cim.Feeder)
network.pprint(cim.Feeder)

In [ ]:
# Print total load served by substation from both feeders
total_load = 0
network.get_all_edges(cim.EnergyConsumer)
for load in network.graph[cim.EnergyConsumer].values():
    total_load = total_load + float(load.p)

print(f'total load is {total_load/1000} kW')

In [ ]:
utils.get_all_data(network)

In [ ]:
utils.write_xml(network, '../test_output/single_bus_and_feeders.xml')